# Two Tests, One Rare Disease

**Topic:** Bayes' rule &nbsp;·&nbsp; **Difficulty:** Easy

A screening test is "99% accurate" and a patient tests positive. The intuitive
answer, a 99% chance they are ill, is off by a factor of about six,
because the disease is rare and false positives outnumber true ones.

Then a second, independent test comes back positive too. Now the arithmetic
swings the other way, and hard.

## The data (created in the next cell)

- `prevalence = 0.002`.
- Test A: sensitivity `0.99`, specificity `0.99`.
- Test B: sensitivity `0.95`, specificity `0.97`, conditionally independent of A
  given the true state.

## Your task

Compute, with plain arithmetic or NumPy:

1. `p_one`: P(ill | A positive).
2. `p_two`: P(ill | A positive **and** B positive). Use the posterior after A as
   the prior for B. Conditional independence means you may multiply the
   likelihoods.
3. `n_false_per_true`: among people who test positive on A alone, the ratio of
   false positives to true positives.
4. Set `answers` to a dict with `"p_one"`, `"p_two"`, `"n_false_per_true"`,
   each a plain number (a scalar, not an array).

## Evaluation

All three are compared to a relative tolerance of 1e-6.

In [ ]:
prevalence = 0.002
sens_a, spec_a = 0.99, 0.99
sens_b, spec_b = 0.95, 0.97
print(f"prevalence {prevalence}, test A {sens_a}/{spec_a}, test B {sens_b}/{spec_b}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the numbers created above to compute the probabilities, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("p_one", [], 0.1655518394648828, -0.027719671348888635, 1e-06),
    ("p_two", [], 0.8626857457347273, -0.14444638867458853, 1e-06),
    ("n_false_per_true", [], 5.040404040404045, -0.8439552463882339, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

In 100,000 people, 200 are ill and 198 of them test positive on A. Of the 99,800
healthy, 998 also test positive. So a positive A means about 198/1196 = 16.6%,
five false alarms for every real case. The second test multiplies the odds by its
own likelihood ratio (0.95/0.03 = 31.7), which takes the posterior odds from
0.198 to 6.3, i.e. about 86%.

```python
prior_odds = prevalence / (1 - prevalence)

lr_a = sens_a / (1 - spec_a)
lr_b = sens_b / (1 - spec_b)

odds_one = prior_odds * lr_a
odds_two = odds_one * lr_b

p_one = odds_one / (1 + odds_one)
p_two = odds_two / (1 + odds_two)

true_pos = prevalence * sens_a
false_pos = (1 - prevalence) * (1 - spec_a)

answers = {
    "p_one": float(p_one),
    "p_two": float(p_two),
    "n_false_per_true": float(false_pos / true_pos),
}
```